In [1]:
"""
Knowledge Distillation — Script 5: Online Distillation (Best Config Only)
==========================================================================
Model   : ResNet-50 × 2 peers (same modified CIFAR-10 arch)
Dataset : CIFAR-10
Method  : DML — Deep Mutual Learning (Zhang 2018), N=2, tau=3.0, epochs=15

Each model i minimises:
  L_i = CE(y, p_i) + (1/(N-1)) * Σ_{j≠i} KL(p_j || p_i)
All peer KL terms use detached probabilities so only model i's graph is
active during model i's backward pass. Each peer has its own GradScaler.

WHY WARM-START FROM BASELINE?
  Training from random init with lr=0.1 collapses both peers to ~0.10
  (random chance) in every run. The KL term between two equally-confused
  peers provides zero signal, so the CE gradient alone must escape a
  saddle point — which cosine decay prevents by shrinking lr too fast.
  Initialising from the pre-trained baseline sidesteps this entirely:
  the peers start with meaningful representations and the KL term
  immediately provides useful soft-label signal.

  Both peers are initialised from the same checkpoint but diverge during
  fine-tuning due to different SGD noise (shuffle + weight decay).

Output : __5__KD_online_distillation_dml_best.json
"""

import os, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch  {torch.__version__}", flush=True)
print(f"[init] Device   : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    _props = torch.cuda.get_device_properties(0)
    TOTAL_VRAM_GB = _props.total_memory / 1024**3
    print(f"[init] GPU      : {torch.cuda.get_device_name(0)}", flush=True)
    print(f"[init] VRAM     : {TOTAL_VRAM_GB:.1f} GB", flush=True)
else:
    TOTAL_VRAM_GB = 0.0

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../../__2__baseline_metrics.json"
OUTPUT_JSON           = "__5__KD_online_distillation_dml_best_10epochs.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 32
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
USE_AMP     = (DEVICE.type == "cuda")

# Best config
VARIANT   = "dml"
N_PEERS   = 2
TAU       = 3.0
LR        = 0.01    # fine-tuning LR — peers start from pre-trained weights
KD_EPOCHS = 10

print(f"[init] AMP      : {USE_AMP}", flush=True)
print(f"[init] Config   : {VARIANT} | N={N_PEERS} | tau={TAU} | "
      f"lr={LR} | epochs={KD_EPOCHS}", flush=True)
print(f"[init] Init     : warm-start from pre-trained baseline", flush=True)

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)


# ── Model ──────────────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_pretrained_peer(path: str) -> nn.Module:
    """Load a fresh copy of the baseline for use as a peer starting point."""
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    return model


# ── Data ───────────────────────────────────────────────────────────────────────
def get_train_loader() -> DataLoader:
    t = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../../data", train=True, download=True, transform=t)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader() -> DataLoader:
    t = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../../data", train=False, download=True, transform=t)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


# ── Helpers ────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024**2
    finally:
        os.remove(tmp)

def param_count_nonzero(model: nn.Module) -> int:
    return int(sum((p != 0).sum().item() for p in model.parameters()))

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        preds.extend(model(inputs).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    yp, yt = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(yt, yp)),
        "precision": float(precision_score(yt, yp, average="macro", zero_division=0)),
        "recall"   : float(recall_score(yt, yp, average="macro", zero_division=0)),
        "f1"       : float(f1_score(yt, yp, average="macro", zero_division=0)),
    }

def measure_inference_detailed(model: nn.Module, device: torch.device) -> dict:
    model  = copy.deepcopy(model).eval().to(device)
    is_gpu = device.type == "cuda"

    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)

    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }

def compute_flops(model: nn.Module) -> float:
    from thop import profile as thop_profile
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None

def evaluate_ensemble(peers: list, loader: DataLoader) -> float:
    for p in peers: p.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, lbls in loader:
            inputs = inputs.to(DEVICE, non_blocking=True)
            probs  = torch.stack([
                F.softmax(p(inputs), dim=1).cpu() for p in peers
            ]).mean(0)
            all_preds.extend(probs.argmax(1).numpy())
            all_labels.extend(lbls.numpy())
    return float(accuracy_score(np.array(all_labels), np.array(all_preds)))


# ══════════════════════════════════════════════════════════════════════════════
# DML training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_dml(train_loader: DataLoader, test_loader: DataLoader,
            baseline_acc: float) -> dict:

    print(f"\n  ┌─ [DML / N={N_PEERS} / tau={TAU} / lr={LR} / epochs={KD_EPOCHS}]",
          flush=True)

    try:
        # ── Warm-start both peers from the pre-trained baseline ───────────────
        # deepcopy gives each peer independent weights that diverge under
        # different SGD noise (shuffle randomness + weight-decay gradient).
        print(f"      [init] Loading {N_PEERS} peers from pre-trained baseline ...",
              flush=True)
        peers = [
            load_pretrained_peer(BASELINE_MODEL_PATH).to(DEVICE)
            for _ in range(N_PEERS)
        ]
        print(f"      [init] Both peers warm-started — ready for DML fine-tuning",
              flush=True)

        # Per-peer optimizers, schedulers, scalers
        optimizers = [
            torch.optim.SGD(p.parameters(), lr=LR, momentum=0.9,
                            weight_decay=1e-4, nesterov=True)
            for p in peers
        ]
        schedulers = [
            torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=KD_EPOCHS)
            for opt in optimizers
        ]
        scalers = [torch.amp.GradScaler(enabled=USE_AMP) for _ in peers]

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        peer_epoch_acc = [[] for _ in range(N_PEERS)]
        t_start = time.perf_counter()

        for epoch in range(KD_EPOCHS):
            for p in peers: p.train()
            correct_per_peer = [0] * N_PEERS
            total = 0
            t_ep  = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                # Forward all peers; detach other peers' logits immediately
                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    all_logits    = [p(inputs) for p in peers]
                    all_probs_det = [
                        F.softmax(lg.detach() / TAU, dim=1) for lg in all_logits
                    ]

                # Per-peer backward — no retain_graph needed
                for i in range(N_PEERS):
                    optimizers[i].zero_grad(set_to_none=True)

                    with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                        ce_i = F.cross_entropy(all_logits[i], targets)

                        kl_sum = torch.zeros(1, device=DEVICE)
                        for j in range(N_PEERS):
                            if j == i:
                                continue
                            log_pi = F.log_softmax(all_logits[i] / TAU, dim=1)
                            kl_sum = kl_sum + F.kl_div(
                                log_pi, all_probs_det[j], reduction="batchmean"
                            ) * (TAU ** 2)

                        loss_i = ce_i + kl_sum / max(N_PEERS - 1, 1)

                    scalers[i].scale(loss_i).backward()
                    scalers[i].step(optimizers[i])
                    scalers[i].update()

                with torch.no_grad():
                    for i in range(N_PEERS):
                        correct_per_peer[i] += (
                            all_logits[i].detach().argmax(1) == targets
                        ).sum().item()
                total += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done    = batch_idx + 1
                    eta     = elapsed / done * (len(train_loader) - done)
                    accs    = " | ".join(
                        f"p{i}={correct_per_peer[i]/total:.3f}" for i in range(N_PEERS)
                    )
                    print(f"      [epoch {epoch+1}/{KD_EPOCHS}] "
                          f"batch {done}/{len(train_loader)}  {accs}  ETA={eta:.0f}s",
                          flush=True)

            for sch in schedulers: sch.step()
            sync()

            ep_time   = time.perf_counter() - t_ep
            remaining = KD_EPOCHS - (epoch + 1)
            acc_vals  = [round(correct_per_peer[i] / total, 6) for i in range(N_PEERS)]
            for i in range(N_PEERS):
                peer_epoch_acc[i].append(acc_vals[i])

            print(f"      [epoch {epoch+1}/{KD_EPOCHS}] DONE  "
                  f"peer_accs={acc_vals}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s",
                  flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                        if DEVICE.type == "cuda" else None)

        # ── Evaluation ────────────────────────────────────────────────────────
        print("      [eval] Classification metrics (all peers) ...", flush=True)
        peer_metrics = []
        for i, peer in enumerate(peers):
            m = evaluate(peer, test_loader)
            peer_metrics.append(m)
            print(f"             Peer {i}: acc={m['accuracy']:.4f}", flush=True)

        best_idx     = max(range(N_PEERS), key=lambda i: peer_metrics[i]["accuracy"])
        best_metrics = peer_metrics[best_idx]
        avg_acc      = float(np.mean([m["accuracy"] for m in peer_metrics]))

        print("      [eval] Ensemble accuracy ...", flush=True)
        ens_acc = evaluate_ensemble(peers, test_loader)
        print(f"             Ensemble: acc={ens_acc:.4f}", flush=True)

        print("      [eval] Inference timing (best peer) ...", flush=True)
        inference_info = measure_inference_detailed(peers[best_idx], DEVICE)

        print("      [eval] FLOPs ...", flush=True)
        flops_g = compute_flops(peers[best_idx])

        size_mb   = model_size_mb(peers[best_idx])
        params_nz = param_count_nonzero(peers[best_idx])
        acc_drop  = baseline_acc - best_metrics["accuracy"]
        ens_drop  = baseline_acc - ens_acc

        print(f"  └─ BestPeer={best_metrics['accuracy']:.4f}  "
              f"Ensemble={ens_acc:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : "best_config",
            "variant"           : VARIANT,
            "n_peers"           : N_PEERS,
            "temperature"       : TAU,
            "lr"                : LR,
            "lr_schedule"       : "cosine",
            "init"              : "warm_start_from_baseline",
            "epochs"            : KD_EPOCHS,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "peer_epoch_acc"    : peer_epoch_acc,
            "peer_final_acc"    : [m["accuracy"] for m in peer_metrics],
            "best_peer_idx"     : best_idx,
            "accuracy"          : round(best_metrics["accuracy"],  6),
            "ensemble_accuracy" : round(ens_acc, 6),
            "avg_peer_accuracy" : round(avg_acc, 6),
            "precision"         : round(best_metrics["precision"], 6),
            "recall"            : round(best_metrics["recall"],    6),
            "f1"                : round(best_metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "ensemble_drop"     : round(ens_drop, 6),
            "params_nonzero"    : params_nz,
            "size_mb"           : round(size_mb, 4),
            "flops_G"           : flops_g,
            "inference_ms"      : inference_info,
            "peak_gpu_memory_mb": peak_gpu_mb,
            "description"       : (
                f"Online KD ({VARIANT}, N={N_PEERS}, tau={TAU}, "
                f"cosine, epochs={KD_EPOCHS}, warm-start)"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "variant": VARIANT, "n_peers": N_PEERS, "temperature": TAU,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ───────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Online Distillation — DML (Warm-Start Fix)")
    print(f"  variant={VARIANT} | N={N_PEERS} | tau={TAU} | lr={LR} | epochs={KD_EPOCHS}")
    print(f"  init=warm_start_from_baseline")
    print(f"  device={DEVICE} | batch={BATCH_SIZE} | AMP={USE_AMP}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline acc : {baseline_acc:.4f}\n", flush=True)

    ref        = build_resnet50_cifar(NUM_CLASSES)
    ref_size   = model_size_mb(ref)
    ref_params = param_count_nonzero(ref)
    print(f"  Per-peer — Size: {ref_size:.2f} MB  "
          f"Non-zero params: {ref_params:,}\n", flush=True)

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    result = run_dml(train_loader, test_loader, baseline_acc)

    report = {
        "method"      : "online_distillation_dml_best",
        "description" : (
            f"DML best config: N={N_PEERS} peers, tau={TAU}, lr={LR}, "
            f"epochs={KD_EPOCHS}, cosine LR, warm-start from pre-trained baseline"
        ),
        "model_arch"  : "ResNet-50 × N (CIFAR-10 modified, 3×3 conv1, no maxpool)",
        "dataset"     : "CIFAR-10",
        "train_device": str(DEVICE),
        "use_amp"     : USE_AMP,
        "kd_epochs"   : KD_EPOCHS,
        "baseline"    : baseline,
        "baseline_model_info": {
            "size_mb"       : round(ref_size, 4),
            "params_nonzero": ref_params,
        },
        "result": result,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    if result.get("accuracy") is not None:
        print(f"  BestPeer={result['accuracy']:.4f}  |  "
              f"Ensemble={result['ensemble_accuracy']:.4f}  |  "
              f"Drop={result['accuracy_drop']:+.4f}  |  "
              f"F1={result['f1']:.4f}  |  "
              f"Throughput={result['inference_ms']['throughput_imgs_sec']:.0f} img/s  |  "
              f"FLOPs={result['flops_G']} G")


if __name__ == "__main__":
    main()

[init] PyTorch  2.11.0+cu128
[init] Device   : cuda
[init] GPU      : NVIDIA GeForce RTX 5070 Laptop GPU
[init] VRAM     : 8.0 GB
[init] AMP      : True
[init] Config   : dml | N=2 | tau=3.0 | lr=0.01 | epochs=10
[init] Init     : warm-start from pre-trained baseline

  Online Distillation — DML (Warm-Start Fix)
  variant=dml | N=2 | tau=3.0 | lr=0.01 | epochs=10
  init=warm_start_from_baseline
  device=cuda | batch=128 | AMP=True

  Baseline acc : 0.9320

  Per-peer — Size: 90.03 MB  Non-zero params: 23,494,282


  ┌─ [DML / N=2 / tau=3.0 / lr=0.01 / epochs=10]
      [init] Loading 2 peers from pre-trained baseline ...
      [init] Both peers warm-started — ready for DML fine-tuning
      [epoch 1/10] batch 100/391  p0=0.971 | p1=0.971  ETA=54s
      [epoch 1/10] batch 200/391  p0=0.958 | p1=0.958  ETA=35s
      [epoch 1/10] batch 300/391  p0=0.953 | p1=0.953  ETA=17s
      [epoch 1/10] DONE  peer_accs=[0.95056, 0.95056]  time=71.4s  ETA_remaining=643s
      [epoch 2/10] batch 100/391